In [0]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import current_timestamp, to_utc_timestamp
from pyspark.sql.functions import when, col

In [0]:
dbutils.widgets.text("job_id", "")
job_id = dbutils.widgets.get("job_id")

dbutils.widgets.text("source_file_path", "")
source_file_path = dbutils.widgets.get("source_file_path")

In [0]:
watermark= spark.sql(f"""
SELECT watermark_column 
FROM pt.dev_metadata_schema.tbl_parameters 
where job_id='{job_id}' and source_file_path = '{source_file_path}'
""")
watermark=watermark.collect()[0][0]

In [0]:
if source_file_path =='dim_tables/products':

   products_df = spark.sql(f"""
   SELECT product_id,product_name,units,to_timestamp(feed_date) as feed_date
   FROM pt.dev_bronze.dim_products
   WHERE to_timestamp(feed_date)
      > to_timestamp('{watermark}')
   """)

   products_df = products_df.withColumn(
    "stakeholder",
    when(col("product_id").like("%ST%"), "Denise Rockwell")
    .when(col("product_id").like("%GT%"), "Van Eybergen Paul")
    .otherwise("Unknown")
)

elif source_file_path =='dim_tables/tier_details':

   tier_details_df=spark.sql(f"""select tier_id,tier_name,to_timestamp(feed_date) as feed_date
   FROM pt.dev_bronze.dim_tier_details
   WHERE to_timestamp(feed_date)
      > to_timestamp('{watermark}')
   """)

In [0]:
if source_file_path =='dim_tables/products' and products_df.isEmpty():
    dbutils.notebook.exit("no_data")
if source_file_path =='dim_tables/tier_details' and tier_details_df.isEmpty():
    dbutils.notebook.exit("no_data")

In [0]:
if source_file_path =='dim_tables/products'and not products_df.isEmpty():

    products_df.write.format("delta") \
        .option("path", "abfss://silver@projecttrackingadlsdev.dfs.core.windows.net/dim_products") \
        .mode("append") \
        .saveAsTable("pt.dev_silver.dim_products")

    spark.sql(f"""
        update pt.dev_metadata_schema.tbl_parameters set watermark_column=current_timestamp
        where job_id=103 and source_file_path like '%products%' """)
   

elif source_file_path =='dim_tables/tier_details' and not tier_details_df.isEmpty():

     tier_details_df.write.format("delta") \
        .option("path", "abfss://silver@projecttrackingadlsdev.dfs.core.windows.net/dim_tier_details") \
        .mode("append") \
        .saveAsTable("pt.dev_silver.dim_tier_details")

     spark.sql(f"""
        update pt.dev_metadata_schema.tbl_parameters set watermark_column=current_timestamp
        where job_id=103 and source_file_path like '%tier_details%' """)

In [0]:
dbutils.notebook.exit("loaded")